# 2030 – Model Encoding, Feature Selection, modeling and valuation for each brand

## Notebook Overview
This notebook shows our approach of spliting into brand specific models.

- Inputs: `11_processed_train_data.csv`, `11_processed_test_data.csv`.
- Outputs: `30_submission_YYYYMMDD_HHMM_rf.csv`, `30_submission_YYYYMMDD_HHMM_etr.csv`, `30_submission_YYYYMMDD_HHMM_hgbr.csv` and `30_submission_YYYYMMDD_HHMM_hybrid.csv` in `../data/submissions/`.

### Preprocessing
This notebook first splits the data into validation and training stratified by brand (important for later). It then imputes the categorical missing values. After that it splits the data set into different brands to from now on do steps only for the specific brand. In the next step we impute the numerical missing values. After that we both one hot and target encode the categorical features. We decide not to scale our data, since it is useless for the models we use here.

### Feature Selection
We use filter (variance, correlation) and wrapper (RFE) methods, selecting features based on validation performance while avoiding leakage.

### Models
For the Models we use a Random Forest Regressor, an Extra Tree Regressor and a Hist Gradient Boosting Regressor. For all of them we use parameter grid search. At the end we also combine Extra Tree Regressor and a Hist Gradient Boosting Regressor (with weights), which is our best model we got in this project.

### Evaluation

TBD

Note: All model selection uses the validation split; only the final chosen model is trained on train+val before generating test predictions.

# Introduction

To gain additional insights beyond a single global model, we explored a modeling strategy that accounts for structural differences between car brands. Preliminary analysis (see Appendix) showed that vehicle prices vary substantially across brands. We also assume that cars within the same brand tend to be more comparable to each other than to cars from different brands. This observation motivated the idea of training separate models per brand rather than relying solely on a single model trained on all data.

A second motivation concerns the `model` feature. When modeling all brands jointly, this variable contains a very large number of categories, making it difficult to extract meaningful information. By training brand-specific models, the `model` column becomes much more compact within each subset, allowing for more expressive encodings (e.g., one-hot encoding combined with target encoding instead of target encoding alone), which can potentially improve predictive performance.

In addition to comparing individual models, we also aim to investigate whether combining different tree-based algorithms can further improve performance. Tree ensembles such as Extra Trees and HistGradientBoosting capture complementary aspects of the data, and a hybrid ensemble may benefit from the strengths of both approaches while mitigating their individual weaknesses.

To evaluate these strategies, we trained and compared multiple tree-based algorithms: Random Forest, Extra Trees, HistGradientBoosting, and a hybrid ensemble combining Extra Trees and HistGradientBoosting. All approaches were applied both in a brand-specific setting and in a single global model trained on all brands. The final step consists of comparing these approaches to assess whether the additional complexity of brand-specific and ensemble-based modeling is justified and aligned with the observed results.


# Table of Contents

<a id="top"></a>

- [Notebook Overview](#notebook-overview)
- [1. Import Libraries](#sec-1-imports)
- [1. Functions](#sec-1-imports)
- [2. Load Dataset](#sec-2-load)
- [3. Imputation Strategies](#sec-3-sanity)
- [Intermediate step: Split Brands](#intermediate-step-split-brands)
- [3.2 Numerical Imputation](#3-2-numerical-imputation)
- [5. Feature Engineering](#5-feature-engineering)
- [6. Scaling](#6-scaling)
- [7. Encoding](#7-encoding)
- [8. Feature Selection](#sec-4-filter)
- [9. Models](#9-models)
- [9.2 Extra Trees Regressor](#9-2-extra-trees-regressor)
- [9.3 Hist Gradient Boosting Regressor](#9-3-hist-gradient-boosting-regressor)
- [9.4 Hybrid Model (Extra Tree Regressor * 0,6 + High Gradient Boosting * 0,4)](#9-4-hybrid-model-extra-tree-regressor-0-6-high-gradient-boosting-0-4)


<a id="sec-1-imports"></a>
## 1. Import Libraries


All needed imports sorted by library.

In [91]:
# Standard library
import json
import math
import os
import re
import warnings
from datetime import datetime
from pathlib import Path

# third-party
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv

# sklearn: model selection / metrics
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import ParameterGrid, train_test_split

# sklearn: preprocessing / pipeline
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, RobustScaler, TargetEncoder

# sklearn: models
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

# sklearn: feature selection / base / inspection
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.inspection import permutation_importance

# sklearn: evaluation
from sklearn.metrics import mean_absolute_error, median_absolute_error, mean_squared_error, r2_score

<a id="sec-1-imports"></a>
## 1. Functions


In [92]:
# Function to summarize missing values for a given dataset
def missing_report(X: pd.DataFrame, name: str) -> pd.DataFrame:
    """Generate and display a simple missing-value report for a DataFrame.

    Parameters
    ----------
    X : pd.DataFrame
        Input data for which missing values should be summarized.
    name : str
        A label used in printed output to identify this dataset (e.g. 'train', 'test').

    Returns
    -------
    pd.DataFrame
        A DataFrame with two columns:
        - 'n_missing': absolute number of missing values per column
        - 'pct_missing': percentage of missing values per column

        If no missing values are found, an empty DataFrame with the same columns is returned.
    """
    # Count missing values per column
    mv = X.isna().sum()
    # Keep only columns with at least one missing value, sorted by count
    mv = mv[mv > 0].sort_values(ascending=False)

    # If there are no missing values, print a short message and return an empty report
    if mv.empty:
        print(f"[{name}] No missing values found. (n_rows={len(X)})")
        return pd.DataFrame(columns=["n_missing", "pct_missing"])

    # Compute percentage of missing values per column (rounded to two decimals)
    pct = (mv / len(X) * 100).round(2)

    # Combine counts and percentages into a single DataFrame
    report = pd.DataFrame({"n_missing": mv, "pct_missing": pct})

    # Print and display the report for quick inspection
    print(f"[{name}] Missing Values (n_rows={len(X)}):")
    display(report)

    return report

In [93]:
# Function to build an imputation pipeline template based on available features
def make_template_for(features):
    """Create an imputation pipeline tailored to the given feature set.

    Parameters
    ----------
    features : list-like
        Collection of feature names present in the dataset.

    Returns
    -------
    sklearn.Pipeline
        An imputation pipeline configured with:
        - low-cardinality categorical columns (Brand, fuelType, transmission), if present
        - high-cardinality categorical columns (model), if present
    """
    # Select low-cardinality categorical columns that are actually present
    low_card = [c for c in ["Brand", "fuelType", "transmission"] if c in features]
    # Select high-cardinality categorical columns that are actually present
    high_card = [c for c in ["model"] if c in features]

    # Create a pipeline that knows how to impute low- and high-card categorical features
    return create_imputation_pipeline(low_card_cols=low_card, high_card_cols=high_card)

In [94]:
# Function to apply all fitted imputers in a consistent order
def apply_imputers(X):
    """Apply all pre-fitted imputers to the input data `X` in sequence.

    This function assumes that the global imputers (trans_imputer, fuel_imputer,
    brand_imputer, model_imputer) have already been fitted on the training data.
    It returns the transformed feature matrix with missing values imputed.
    """
    # Apply transmission imputer
    X = trans_imputer.transform(X)
    # Apply fuel type imputer
    X = fuel_imputer.transform(X)
    # Apply brand imputer
    X = brand_imputer.transform(X)
    # Apply model imputer
    X = model_imputer.transform(X)
    return X

In [95]:
# Function to drop low-variance features (below a given threshold)
def drop_low_variance_features(x, threshold):
    """Remove columns whose variance is below `threshold`.
    Returns the reduced frame and the list of dropped features for consistency across splits.
    """
    # Calculate the variance of each feature
    variances = x.var()
    # Identify features with variance below the threshold
    low_variance_features = variances[variances < threshold].index
    # Drop low-variance features
    x = x.drop(columns=low_variance_features)
    return x, low_variance_features

In [96]:
# Helper function to create specific pipelines for each imputer
# This ensures we only try to encode columns that are actually present as features
def create_imputation_pipeline(low_card_cols, high_card_cols=[]):
    transformers = []
    if low_card_cols:
        transformers.append(('low_card', low_cardinality_pipeline, low_card_cols))
    if high_card_cols:
        transformers.append(('high_card', high_cardinality_pipeline, high_card_cols))
        
    # remainder='passthrough' keeps the numerical columns
    encoder = ColumnTransformer(transformers, remainder='passthrough', verbose_feature_names_out=False)
    
    return Pipeline([
        ('encoder', encoder),
        ("rf_model", RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42))
    ])

<a id="sec-2-load"></a>
## 2. Load Dataset


Load the mapped and normalized datasets produced in notebook 11 and confirm shapes before proceeding. <br>

In [97]:
# Load the data paths
data_dir = "../data/"

# Load the raw data into a pandas dataframe
df = pd.read_csv(os.path.join(data_dir, f"processed_data/11_processed_train_data.csv"))
x_test = pd.read_csv(os.path.join(data_dir, f"processed_data/11_processed_test_data.csv"))

# put carID as Index
df.set_index("carID", inplace=True)
x_test.set_index("carID", inplace=True)

print("Loaded shape:", df.shape)
display(df.head(3))

print("Loaded test shape:", x_test.shape)
display(x_test.head(3))

Loaded shape: (73744, 12)


,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,previousOwners,hasDamage
carID,,,,,,,,,,,,
69512,Volkswagen,Golf,2016.0,22290.0,Semi-Auto,28421.0,Petrol,NaN,NaN,2.0,4.0,0.0
53000,Toyota,Yaris,2019.0,13790.0,Manual,4589.0,Petrol,145.0,47.9,1.5,1.0,0.0
6366,Audi,Q2,2019.0,24990.0,Semi-Auto,3624.0,Petrol,145.0,40.9,1.5,4.0,0.0


Loaded test shape: (32567, 11)


,Brand,model,year,transmission,mileage,fuelType,tax,mpg,engineSize,previousOwners,hasDamage
carID,,,,,,,,,,,
89856,Hyundai,i30,NaN,Automatic,30700.0,Petrol,205.0,41.5,1.6,3.0,0.0
106581,Volkswagen,Tiguan,2017.0,Semi-Auto,NaN,Petrol,150.0,38.2,2.0,2.0,0.0
80886,BMW,2 Series,2016.0,Automatic,36792.0,Petrol,125.0,51.4,1.5,2.0,0.0


In the next cell we split our data into train and validation data. We use an 85/15 train–validation split, stratified by Brand.
Because the competition already holds out 20% of the full dataset as test data, we wanted to keep the validation size small so that we don't lose too much training data.

In [98]:
# Separate features and target from the training data
x = df.drop(columns=["price"])
y = df["price"]

# Split df (80% of total data) into training and validation
x = df.drop(columns=["price"])
y = df["price"]

# only for stratify
brand_strat = x["Brand"].fillna("Unknown")  

x_train, x_val, y_train, y_val = train_test_split(
    x,
    y,
    test_size=0.15,
    random_state=42,
    stratify=brand_strat # it trys to split the brands equaly --> good so that we have around the same percentage splits into val and train for each brand
)

print(f"x_train shape: {x_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"x_val shape:   {x_val.shape}")
print(f"y_val shape:   {y_val.shape}")
print(f"x_test shape:  {x_test.shape}")

x_train shape: (62682, 11)
y_train shape: (62682,)
x_val shape:   (11062, 11)
y_val shape:   (11062,)
x_test shape:  (32567, 11)


In the next cell we check what missing values for which type of variables we will have to impute.

In [99]:
# Reports for the splits
missing_report(x_train, "x_train")
missing_report(x_val,   "x_val")
missing_report(x_test,  "x_test")

# check which columns are categorical and which numerical
numeric_cols = x_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = x_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print(f"#numeric_cols: {len(numeric_cols)} → {numeric_cols[:10]}{' ...' if len(numeric_cols) > 10 else ''}")
print(f"#categorical_cols: {len(categorical_cols)} → {categorical_cols[:10]}{' ...' if len(categorical_cols) > 10 else ''}")

[x_train] Missing Values (n_rows=62682):


,n_missing,pct_missing
mpg,7825,12.48
tax,7098,11.32
engineSize,2043,3.26
previousOwners,1876,2.99
transmission,1850,2.95
mileage,1839,2.93
model,1420,2.27
hasDamage,1279,2.04
fuelType,1262,2.01
Brand,1251,2.00


[x_val] Missing Values (n_rows=11062):


,n_missing,pct_missing
mpg,1441,13.03
tax,1303,11.78
transmission,355,3.21
engineSize,350,3.16
previousOwners,336,3.04
mileage,324,2.93
model,268,2.42
hasDamage,225,2.03
Brand,221,2.00
fuelType,198,1.79


[x_test] Missing Values (n_rows=32567):


,n_missing,pct_missing
mpg,3985,12.24
tax,3624,11.13
engineSize,1037,3.18
mileage,1009,3.10
year,1007,3.09
transmission,968,2.97
previousOwners,936,2.87
model,751,2.31
fuelType,656,2.01
Brand,649,1.99


#numeric_cols: 7 → ['year', 'mileage', 'tax', 'mpg', 'engineSize', 'previousOwners', 'hasDamage']
#categorical_cols: 4 → ['Brand', 'model', 'transmission', 'fuelType']


<a id="sec-3-sanity"></a>
## 3. Imputation Strategies

We will handle missing values using specific strategies for categorical and numerical features. Inbetween we split our data set into the different brands, more about that later.
We define custom imputers that can be integrated into a pipeline.

### 3.1 Categorical Imputation
Strategies for: `transmission`, `fuelType`, `Brand`, `model`.
These imputers use a combination of rule-based logic (e.g., mode per model) and Random Forrest to fill missing values.

In [100]:
# Used for low-cardinality categorical features
low_cardinality_pipeline = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Used for high-cardinality categorical features like model
high_cardinality_pipeline = Pipeline([
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])
class TransmissionImputer(BaseEstimator, TransformerMixin):
    def __init__(self, pipeline_template, min_model_count=10):
        self.pipeline_template = pipeline_template
        self.min_model_count = min_model_count

    def fit(self, X, y=None):
        df = X.copy()

        # store lookup tables learned from training data
        counts = df.dropna(subset=['transmission'])['model'].value_counts()
        self.valid_models_ = counts[counts >= self.min_model_count].index

        self.model_modes_ = (
            df.dropna(subset=['model', 'transmission'])
              .groupby('model')['transmission']
              .agg(lambda s: s.mode().iloc[0])
        )

        # Train ML model for remaining missing values
        train_df = df[df['transmission'].notna()].copy()

        features = [
            "Brand", "model", "fuelType",
            "engineSize", "year", "mpg", "tax",
        ]

        pipe = clone(self.pipeline_template)
        pipe.fit(train_df[features], train_df['transmission'])

        self.ml_model_ = pipe
        self.features_ = features
        return self

    def transform(self, X):
        df = X.copy()

        # rule-based fill
        for model in self.valid_models_:
            mask = (df['model'] == model) & (df['transmission'].isna())
            if model in self.model_modes_:
                df.loc[mask, 'transmission'] = self.model_modes_[model]

        # ML fallback
        missing_mask = df['transmission'].isna()
        if missing_mask.any():
            df.loc[missing_mask, 'transmission'] = self.ml_model_.predict(
                df.loc[missing_mask, self.features_]
            )
        return df

class FuelTypeImputer(BaseEstimator, TransformerMixin):
    def __init__(self, pipeline_template, min_model_count=10):
        self.pipeline_template = pipeline_template
        self.min_model_count = min_model_count

    def fit(self, X, y=None):
        df = X.copy()

        counts = df.dropna(subset=['fuelType'])['model'].value_counts()
        self.valid_models_ = counts[counts >= self.min_model_count].index

        self.model_modes_ = (
            df.dropna(subset=['model', 'fuelType'])
              .groupby('model')['fuelType']
              .agg(lambda s: s.mode().iloc[0])
        )

        features = ['Brand', 'model', 'transmission',
                    'engineSize', 'year', 'mpg']
        train_df = df[df['fuelType'].notna()]

        pipe = clone(self.pipeline_template)
        pipe.fit(train_df[features], train_df['fuelType'])

        self.ml_model_ = pipe
        self.features_ = features
        return self

    def transform(self, X):
        df = X.copy()

        # rule-based fill
        for model in self.valid_models_:
            mask = (df['model'] == model) & (df['fuelType'].isna())
            df.loc[mask, 'fuelType'] = self.model_modes_.get(model, np.nan)

        # ML fallback
        missing = df['fuelType'].isna()
        if missing.any():
            df.loc[missing, 'fuelType'] = self.ml_model_.predict(
                df.loc[missing, self.features_]
            )
        return df

class BrandImputer(BaseEstimator, TransformerMixin):
    def __init__(self, pipeline_template, min_model_count=20):
        self.pipeline_template = pipeline_template
        self.min_model_count = min_model_count

    def fit(self, X, y=None):
        df = X.copy()

        # rule-based brand per model
        self.model_to_brand_ = (
            df.dropna(subset=['Brand', 'model'])
              .groupby('model')['Brand']
              .agg(lambda s: s.mode().iloc[0])
        )

        # ML fallback
        features = ['transmission', 'engineSize',
                    'fuelType', 'mpg', 'model']
        train_df = df[df['Brand'].notna()]

        pipe = clone(self.pipeline_template)
        pipe.fit(train_df[features], train_df['Brand'])

        self.ml_model_ = pipe
        self.features_ = features
        return self

    def transform(self, X):
        df = X.copy()

        # rule-based fill
        mask = df['Brand'].isna() & df['model'].notna()
        df.loc[mask, 'Brand'] = df.loc[mask, 'model'].map(self.model_to_brand_)

        # ML fallback
        missing = df['Brand'].isna()
        if missing.any():
            df.loc[missing, 'Brand'] = self.ml_model_.predict(
                df.loc[missing, self.features_]
            )
        return df

class ModelImputer(BaseEstimator, TransformerMixin):
    def __init__(self, pipeline_template, min_brand_count=20):
        self.pipeline_template = pipeline_template
        self.min_brand_count = min_brand_count

    def fit(self, X, y=None):
        df = X.copy()

        # rule-based: most frequent model per (Brand, transmission)
        self.lookup_ = (
            df.dropna(subset=['Brand', 'transmission', 'model'])
              .groupby(['Brand', 'transmission'])['model']
              .agg(lambda s: s.value_counts().idxmax())
        )

        # ML fallback
        features = ['Brand', 'year', 'engineSize', 'mpg',
                    'tax', 'mileage', 'fuelType', 'transmission']
        train_df = df[df['model'].notna()]

        pipe = clone(self.pipeline_template)
        pipe.fit(train_df[features], train_df['model'])

        self.ml_model_ = pipe
        self.features_ = features
        return self

    def transform(self, X):
        df = X.copy()

        # rule-based fill
        for idx, row in df[df['model'].isna()].iterrows():
            key = (row['Brand'], row['transmission'])
            if key in self.lookup_.index:
                df.at[idx, 'model'] = self.lookup_.loc[key]

        # ML fallback
        missing = df['model'].isna()
        if missing.any():
            df.loc[missing, 'model'] = self.ml_model_.predict(
                df.loc[missing, self.features_]
            )
        return df

In the next cell we impute the categorical values for train, val and test data. We only use train data to fit the imputer.

In [101]:
# Features wie in deinen Klassen
trans_features = ["Brand","model","fuelType","engineSize","year","mpg","tax"]
fuel_features  = ["Brand","model","transmission","engineSize","year","mpg"]
brand_features = ["transmission","engineSize","fuelType","mpg","model"]
model_features = ["Brand","year","engineSize","mpg","tax","mileage","fuelType","transmission"]

# Templates je Imputer (nur relevante Spalten werden encoded)
trans_imputer = TransmissionImputer(make_template_for(trans_features))
fuel_imputer  = FuelTypeImputer(make_template_for(fuel_features))
brand_imputer = BrandImputer(make_template_for(brand_features))
model_imputer = ModelImputer(make_template_for(model_features))

# Fit nur auf TRAIN
for imp in [trans_imputer, fuel_imputer, brand_imputer, model_imputer]:
    imp.fit(x_train)

x_train_imp = apply_imputers(x_train)
x_val_imp   = apply_imputers(x_val)
x_test_imp  = apply_imputers(x_test)

The next cell just checks if we successfully removed the categorical missing values, which we did.

In [102]:
missing_report(x_train_imp, "x_train_imp")
missing_report(x_val_imp,   "x_val_imp")
missing_report(x_test_imp,  "x_test_imp")

[x_train_imp] Missing Values (n_rows=62682):


,n_missing,pct_missing
mpg,7825,12.48
tax,7098,11.32
engineSize,2043,3.26
previousOwners,1876,2.99
mileage,1839,2.93
hasDamage,1279,2.04


[x_val_imp] Missing Values (n_rows=11062):


,n_missing,pct_missing
mpg,1441,13.03
tax,1303,11.78
engineSize,350,3.16
previousOwners,336,3.04
mileage,324,2.93
hasDamage,225,2.03


[x_test_imp] Missing Values (n_rows=32567):


,n_missing,pct_missing
mpg,3985,12.24
tax,3624,11.13
engineSize,1037,3.18
mileage,1009,3.10
year,1007,3.09
previousOwners,936,2.87
hasDamage,597,1.83


,n_missing,pct_missing
mpg,3985,12.24
tax,3624,11.13
engineSize,1037,3.18
mileage,1009,3.10
year,1007,3.09
previousOwners,936,2.87
hasDamage,597,1.83


## Intermediate step: Split Brands

After having no missing values in brands left, we can split the data sets into the different brands. In the next cell we check the distribution of brands in our train data.

In [103]:
x_train_imp['Brand'].value_counts()

Brand
Ford             13519
Mercedes-Benz     9825
Volkswagen        8736
Opel              7872
BMW               6229
Audi              6165
Toyota            3900
Škoda             3618
Hyundai           2818
Name: count, dtype: int64

We now split the train data into the brands seen in the cell before, we also need to split the y_train values.

In [104]:
# assumes: x_train and y_train have matching indices
brand_splits = {}

for brand, X_b in x_train_imp.groupby("Brand"):
    y_b = y_train.loc[X_b.index]   # pick same rows in y
    brand_splits[brand] = (X_b, y_b)

x_train_ford, y_train_ford = brand_splits["Ford"]
x_train_merc, y_train_merc = brand_splits["Mercedes-Benz"]
x_train_volk, y_train_volk = brand_splits["Volkswagen"]
x_train_opel, y_train_opel = brand_splits["Opel"]
x_train_bmw, y_train_bmw = brand_splits["BMW"]
x_train_audi, y_train_audi = brand_splits["Audi"]
x_train_toyo, y_train_toyo = brand_splits["Toyota"]
x_train_skod, y_train_skod = brand_splits["Škoda"]
x_train_hyun, y_train_hyun = brand_splits["Hyundai"]

We do the same for validation data.

In [105]:
# assumes: x_train and y_train have matching indices
brand_splits = {}

for brand, X_b in x_val_imp.groupby("Brand"):
    y_b = y_val.loc[X_b.index]   # pick same rows in y
    brand_splits[brand] = (X_b, y_b)

x_val_ford, y_val_ford = brand_splits["Ford"]
x_val_merc, y_val_merc = brand_splits["Mercedes-Benz"]
x_val_volk, y_val_volk = brand_splits["Volkswagen"]
x_val_opel, y_val_opel = brand_splits["Opel"]
x_val_bmw, y_val_bmw = brand_splits["BMW"]
x_val_audi, y_val_audi = brand_splits["Audi"]
x_val_toyo, y_val_toyo = brand_splits["Toyota"]
x_val_skod, y_val_skod = brand_splits["Škoda"]
x_val_hyun, y_val_hyun = brand_splits["Hyundai"]

And also for test data.

In [106]:
brand_splits = {}

for brand, X_b in x_test_imp.groupby("Brand"):
    brand_splits[brand] = X_b

x_test_ford = brand_splits["Ford"]
x_test_merc = brand_splits["Mercedes-Benz"]
x_test_volk = brand_splits["Volkswagen"]
x_test_opel = brand_splits["Opel"]
x_test_bmw = brand_splits["BMW"]
x_test_audi = brand_splits["Audi"]
x_test_toyo = brand_splits["Toyota"]
x_test_skod = brand_splits["Škoda"]
x_test_hyun = brand_splits["Hyundai"]

With those dataframes we build list, so that we can iterate over them.

In [107]:
train_x_list = [x_train_ford, x_train_merc, x_train_volk, x_train_opel, x_train_bmw, 
x_train_audi, x_train_toyo, x_train_skod, x_train_hyun]

train_y_list = [y_train_ford, y_train_merc, y_train_volk, y_train_opel, y_train_bmw, 
y_train_audi, y_train_toyo, y_train_skod, y_train_hyun]

val_x_list = [x_val_ford, x_val_merc, x_val_volk, x_val_opel, x_val_bmw, 
x_val_audi, x_val_toyo, x_val_skod, x_val_hyun]

val_y_list = [y_val_ford, y_val_merc, y_val_volk, y_val_opel, y_val_bmw, 
y_val_audi, y_val_toyo, y_val_skod, y_val_hyun]

test_x_list = [x_test_ford, x_test_merc, x_test_volk, x_test_opel, x_test_bmw, 
x_test_audi, x_test_toyo, x_test_skod, x_test_hyun]

We then drop the brand column, because its useless after the split.

In [108]:
train_x_list = [df.drop(columns=['Brand']) for df in train_x_list]

val_x_list = [df.drop(columns=['Brand']) for df in val_x_list]

test_x_list = [df.drop(columns=['Brand']) for df in test_x_list]

## 3.2 Numerical Imputation
Strategies for: `mileage`, `year`, `mpg`, `tax`, `engineSize`, `previousOwners`.
These imputers primarily use median values, grouped by relevant features (e.g., median mileage per year) to ensure realistic fills.

In [109]:
class MileageImputer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        df = X.copy()
        self.year_medians_ = df.groupby('year')['mileage'].median()
        self.global_median_ = df['mileage'].median()
        return self

    def transform(self, X):
        df = X.copy()
        df['mileage'] = df['mileage'].fillna(df['year'].map(self.year_medians_))
        df['mileage'] = df['mileage'].fillna(self.global_median_)
        return df


class YearImputer(BaseEstimator, TransformerMixin):
    def __init__(self, bins=30):
        self.bins = bins

    def fit(self, X, y=None):
        df = X.copy()
        df['mileage_bin'] = pd.cut(df['mileage'], bins=self.bins)
        self.bin_medians_ = df.groupby('mileage_bin')['year'].median()
        self.global_median_ = df['year'].median()
        return self

    def transform(self, X):
        df = X.copy()
        df['mileage_bin'] = pd.cut(df['mileage'], bins=self.bins)
        df['year'] = df['year'].fillna(df['mileage_bin'].map(self.bin_medians_))
        df['year'] = df['year'].fillna(self.global_median_)
        df.drop(columns='mileage_bin', inplace=True)
        return df


class MPGImputer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        df = X.copy()
        # ONLY model + fuelType
        self.model_medians_ = df.groupby(['model', 'fuelType'])['mpg'].median()
        self.fuel_medians_ = df.groupby('fuelType')['mpg'].median()
        self.global_median_ = df['mpg'].median()
        return self

    def transform(self, X):
        df = X.copy()

        for idx, row in df[df['mpg'].isna()].iterrows():
            km = (row['model'], row['fuelType'])
            if km in self.model_medians_.index:
                df.at[idx, 'mpg'] = self.model_medians_.loc[km]

        df['mpg'] = df['mpg'].fillna(df['fuelType'].map(self.fuel_medians_))
        df['mpg'] = df['mpg'].fillna(self.global_median_)
        return df


class PreviousOwnersImputer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.median_ = X['previousOwners'].median()
        return self

    def transform(self, X):
        df = X.copy()
        df['previousOwners'] = df['previousOwners'].fillna(self.median_)
        return df


class TaxImputer(BaseEstimator, TransformerMixin):
    def __init__(self, bins=15):
        self.bins = bins

    def fit(self, X, y=None):
        df = X.copy()
        df['mpg_bin'] = pd.cut(df['mpg'], bins=self.bins)

        # remove Brand level, keep model+fuel+bin and fuel+bin
        self.medians_one_ = df.groupby(['model', 'fuelType', 'mpg_bin'])['tax'].median()
        self.medians_two_ = df.groupby(['fuelType', 'mpg_bin'])['tax'].median()
        self.fuel_medians_ = df.groupby('fuelType')['tax'].median()
        self.global_median_ = df['tax'].median()
        return self

    def transform(self, X):
        df = X.copy()
        df['mpg_bin'] = pd.cut(df['mpg'], bins=self.bins)

        for idx, row in df[df['tax'].isna()].iterrows():
            km = (row['model'], row['fuelType'], row['mpg_bin'])
            ke = (row['fuelType'], row['mpg_bin'])

            if km in self.medians_one_.index:
                df.at[idx, 'tax'] = self.medians_one_.loc[km]
            elif ke in self.medians_two_.index:
                df.at[idx, 'tax'] = self.medians_two_.loc[ke]

        df['tax'] = df['tax'].fillna(df['fuelType'].map(self.fuel_medians_))
        df['tax'] = df['tax'].fillna(self.global_median_)
        df.drop(columns='mpg_bin', inplace=True)
        return df


class EngineSizeImputer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        df = X.copy()
        # ONLY model + fuelType
        self.model_medians_ = df.groupby(['model', 'fuelType'])['engineSize'].median()
        self.model_only_medians_ = df.groupby('model')['engineSize'].median()
        self.global_median_ = df['engineSize'].median()
        return self

    def transform(self, X):
        df = X.copy()

        for idx, row in df[df['engineSize'].isna()].iterrows():
            km = (row['model'], row['fuelType'])
            if km in self.model_medians_.index:
                df.at[idx, 'engineSize'] = self.model_medians_.loc[km]

        df['engineSize'] = df['engineSize'].fillna(df['model'].map(self.model_only_medians_))
        df['engineSize'] = df['engineSize'].fillna(self.global_median_)
        return df

Now we create the pipeline to impute the last missing values.

In [110]:
rule_imputer = Pipeline([
    ("mileage_imp", MileageImputer()),
    ("year_imp", YearImputer()),
    ("mpg_imp", MPGImputer()),
    ("previousOwners_imp", PreviousOwnersImputer()),
    ("tax_imp", TaxImputer()),
    ("engineSize_imp", EngineSizeImputer()),
])

We then train this imputers on train data of the specific brand and then impute the train, val and test data.

In [111]:
for i in range(len(train_x_list)):

    # FIT nur auf TRAIN
    rule_imputer.fit(train_x_list[i])

    # TRANSFORM auf alle Splits
    train_x_list[i] = rule_imputer.transform(train_x_list[i])
    val_x_list[i]   = rule_imputer.transform(val_x_list[i])
    test_x_list[i]  = rule_imputer.transform(test_x_list[i])

C:\Users\Oliver\AppData\Local\Temp\ipykernel_984\2723973335.py:22: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  self.bin_medians_ = df.groupby('mileage_bin')['year'].median()
C:\Users\Oliver\AppData\Local\Temp\ipykernel_984\2723973335.py:77: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  self.medians_one_ = df.groupby(['model', 'fuelType', 'mpg_bin'])['tax'].median()
C:\Users\Oliver\AppData\Local\Temp\ipykernel_984\2723973335.py:78: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=Tr

Because hasDamage only has the value 0 and nan, we set nan to 1 to see if atleast this holds a different value, otherwise it will be useless.

In [112]:
for i in range(len(train_x_list)):
    train_x_list[i]["hasDamage"] = train_x_list[i]["hasDamage"].fillna(1)
    val_x_list[i]["hasDamage"] = val_x_list[i]["hasDamage"].fillna(1)
    test_x_list[i]["hasDamage"] = test_x_list[i]["hasDamage"].fillna(1)

After that we check, if all missing values are gone.

In [113]:
for i in range(len(train_x_list)):
    missing_report(train_x_list[i], f"train_x: {i}")
    missing_report(val_x_list[i],   f"val_x: {i}")
    missing_report(test_x_list[i],  f"test_x: {i}")

[train_x: 0] No missing values found. (n_rows=13519)
[val_x: 0] No missing values found. (n_rows=2390)
[test_x: 0] No missing values found. (n_rows=7027)
[train_x: 1] No missing values found. (n_rows=9825)
[val_x: 1] No missing values found. (n_rows=1732)
[test_x: 1] No missing values found. (n_rows=5108)
[train_x: 2] No missing values found. (n_rows=8736)
[val_x: 2] No missing values found. (n_rows=1552)
[test_x: 2] No missing values found. (n_rows=4550)
[train_x: 3] No missing values found. (n_rows=7872)
[val_x: 3] No missing values found. (n_rows=1399)
[test_x: 3] No missing values found. (n_rows=4090)
[train_x: 4] No missing values found. (n_rows=6229)
[val_x: 4] No missing values found. (n_rows=1092)
[test_x: 4] No missing values found. (n_rows=3228)
[train_x: 5] No missing values found. (n_rows=6165)
[val_x: 5] No missing values found. (n_rows=1083)
[test_x: 5] No missing values found. (n_rows=3204)
[train_x: 6] No missing values found. (n_rows=3900)
[val_x: 6] No missing values 

## 5. Feature Engineering

For feature enginnering, we try to not create to much new noise for our models, we just add a few features. He we declare the class to use them.

In [114]:
class FeatureEngineeringTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        df = X.copy()
        # nur was wir wirklich brauchen
        self.trans_medians_ = df.groupby('transmission')['mpg'].median()
        self.mileage_q75_ = df['mileage'].quantile(0.75)
        self.mileage_q90_ = df['mileage'].quantile(0.90)
        return self

    def transform(self, X):
        df = X.copy()

        # mpg diff vs transmission-median (robust)
        mpg_median_trans = df['transmission'].map(self.trans_medians_)
        mpg_median_trans = mpg_median_trans.fillna(self.trans_medians_.median())
        df['mpg_diff_transmission'] = (df['mpg'] - mpg_median_trans).fillna(0)

        # car age (falls year fehlt vorher imputen)
        df['car_age'] = 2020 - df['year']

        # simple efficiency proxy
        df['efficiency_ratio'] = df['mpg'] / (df['engineSize'] + 0.1)

        # mileage per year
        df['mileage_per_year'] = df['mileage'] / (df['car_age'] + 0.1)

        # high mileage flags
        df['high_mileage_flag'] = (df['mileage'] > self.mileage_q75_).astype(int)
        df['very_high_mileage_flag'] = (df['mileage'] > self.mileage_q90_).astype(int)

        return df

Here we apply those features to our dataframes.

In [115]:
for i in range(len(train_x_list)):

    # FIT nur auf TRAIN
    fe = FeatureEngineeringTransformer()
    fe.fit(train_x_list[i])
    # TRANSFORM auf alle Splits
    train_x_list[i] = fe.transform(train_x_list[i])
    val_x_list[i]   = fe.transform(val_x_list[i])
    test_x_list[i]  = fe.transform(test_x_list[i])

## 6. Scaling

For the models in this notebook, we decided to not use scaling, since it didnt change the score significantly (score was around 1 MAE worse).

In [116]:
"""
for i in range(len(train_x_list)):
    num_cols = train_x_list[i].select_dtypes(include="number").columns
    non_num_cols = train_x_list[i].columns.difference(num_cols)

    scaler = RobustScaler()
    scaler.fit(train_x_list[i][num_cols])

    train_scaled = scaler.transform(train_x_list[i][num_cols])
    val_scaled   = scaler.transform(val_x_list[i][num_cols])
    test_scaled  = scaler.transform(test_x_list[i][num_cols])

    train_x_list[i] = pd.concat([
        pd.DataFrame(train_scaled, columns=num_cols, index=train_x_list[i].index),
        train_x_list[i][non_num_cols]
    ], axis=1)[train_x_list[i].columns]

    val_x_list[i] = pd.concat([
        pd.DataFrame(val_scaled, columns=num_cols, index=val_x_list[i].index),
        val_x_list[i][non_num_cols]
    ], axis=1)[val_x_list[i].columns]

    test_x_list[i] = pd.concat([
        pd.DataFrame(test_scaled, columns=num_cols, index=test_x_list[i].index),
        test_x_list[i][non_num_cols]
    ], axis=1)[test_x_list[i].columns]
"""

'\nfor i in range(len(train_x_list)):\n    num_cols = train_x_list[i].select_dtypes(include="number").columns\n    non_num_cols = train_x_list[i].columns.difference(num_cols)\n\n    scaler = RobustScaler()\n    scaler.fit(train_x_list[i][num_cols])\n\n    train_scaled = scaler.transform(train_x_list[i][num_cols])\n    val_scaled   = scaler.transform(val_x_list[i][num_cols])\n    test_scaled  = scaler.transform(test_x_list[i][num_cols])\n\n    train_x_list[i] = pd.concat([\n        pd.DataFrame(train_scaled, columns=num_cols, index=train_x_list[i].index),\n        train_x_list[i][non_num_cols]\n    ], axis=1)[train_x_list[i].columns]\n\n    val_x_list[i] = pd.concat([\n        pd.DataFrame(val_scaled, columns=num_cols, index=val_x_list[i].index),\n        val_x_list[i][non_num_cols]\n    ], axis=1)[val_x_list[i].columns]\n\n    test_x_list[i] = pd.concat([\n        pd.DataFrame(test_scaled, columns=num_cols, index=test_x_list[i].index),\n        test_x_list[i][non_num_cols]\n    ], axis

## 7. Encoding

For encoding we decided to try to use both Target Encoding and One Hot Encoding next to each other. We also decided to look at the different models for each brand and how many entities we have per model. 

In [117]:
for df in train_x_list:
    print("\n")
    print(df["model"].value_counts())



model
Focus                    5961
Fiesta                   3720
Kuga                     1222
EcoSport                  668
C-MAX                     311
Ka+                       295
Mondeo                    291
B-MAX                     204
S-MAX                     167
Galaxy                    130
Grand C-MAX               127
Ka                        126
Edge                      120
Puma                       42
Tourneo Custom             42
Mustang                    32
Grand Tourneo Connect      31
Tourneo Connect            16
Fusion                     12
StreetKa                    2
Name: count, dtype: int64


model
C-Class      4512
A-Class      1445
E-Class      1100
GLC-Class     552
GLA-Class     463
B-Class       348
GLE-Class     267
CL-Class      265
SL-Class      161
CLS-Class     127
V-Class       118
S-Class       116
GL-Class       74
SLK-Class      55
CLA-Class      54
X-Class        52
GLS-Class      48
M-Class        42
GLB-Class      12
G-Class         

We saw that we have a few models, that dont have that many appearances. To prevent noise and overfitting, we decided to cluster all models under 20 appearances to other. 
A threshold of 20 provided a good balance:
 - It removed extremely rare categories that would otherwise behave like noise,
 - While still preserving the majority of meaningful model groups.

In [118]:
min_count = 20 # first run 100
for i in range(0, len(train_x_list)):
    freq = train_x_list[i]["model"].value_counts()
    rare_models = freq[freq < min_count].index
    
    train_x_list[i]["model"] = train_x_list[i]["model"].replace(rare_models, "Other")

    val_x_list[i]["model"] = val_x_list[i]["model"].replace(rare_models, "Other")

    test_x_list[i]["model"] = test_x_list[i]["model"].replace(rare_models, "Other")

    

After that we perform the encoding.

In [119]:
onehot_cols = ['Brand', 'fuelType', 'transmission', 'model', 'brand_segment', 'car_segment']

# optional: explizit festlegen, welche Spalten per Target Encoding laufen sollen
target_cols = onehot_cols  # oder z.B. ['model', 'Brand'] bei High-Cardinality-Spalten

train_x_encoded_list = []
val_x_encoded_list   = []
test_x_encoded_list  = []

for i in range(len(train_x_list)):
    X_train_i = train_x_list[i]
    X_val_i   = val_x_list[i]
    X_test_i  = test_x_list[i]
    y_train_i = train_y_list[i]   # <- wichtig: Target für das Target Encoding

    # nur die Spalten nehmen, die wirklich existieren
    onehot_cols_i  = [c for c in onehot_cols if c in X_train_i.columns]
    target_cols_i  = [c for c in target_cols if c in X_train_i.columns]

    onehot_encoder = OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False
    )

    # einfache Konfiguration; smooth
    target_encoder = TargetEncoder(
        target_type="continuous",              
        random_state=42
    )

    # One-Hot und TargetEncoding parallel in einem ColumnTransformer
    encoding_transformer = ColumnTransformer(
        transformers=[
            ('onehot', onehot_encoder, onehot_cols_i),
            ('target', target_encoder, target_cols_i),
        ],
        remainder='passthrough',
        verbose_feature_names_out=False
    )

    print(f"\n[{i}] fitting encoder on train_x_list[{i}] "
          f"with onehot: {onehot_cols_i} and target-enc: {target_cols_i}")

    # WICHTIG: y mitgeben, damit TargetEncoder funktioniert
    encoding_transformer.fit(X_train_i, y_train_i)

    # transformieren
    train_arr = encoding_transformer.transform(X_train_i)
    val_arr   = encoding_transformer.transform(X_val_i)
    test_arr  = encoding_transformer.transform(X_test_i)

    feature_names = encoding_transformer.get_feature_names_out()

    # zurück zu DataFrames
    X_train_enc = pd.DataFrame(train_arr, columns=feature_names, index=X_train_i.index)
    X_val_enc   = pd.DataFrame(val_arr,   columns=feature_names, index=X_val_i.index)
    X_test_enc  = pd.DataFrame(test_arr,  columns=feature_names, index=X_test_i.index)

    train_x_encoded_list.append(X_train_enc)
    val_x_encoded_list.append(X_val_enc)
    test_x_encoded_list.append(X_test_enc)

print("Encoding (One-Hot + TargetEncoding) für alle Listeneinträge fertig.")


[0] fitting encoder on train_x_list[0] with onehot: ['fuelType', 'transmission', 'model'] and target-enc: ['fuelType', 'transmission', 'model']

[1] fitting encoder on train_x_list[1] with onehot: ['fuelType', 'transmission', 'model'] and target-enc: ['fuelType', 'transmission', 'model']

[2] fitting encoder on train_x_list[2] with onehot: ['fuelType', 'transmission', 'model'] and target-enc: ['fuelType', 'transmission', 'model']

[3] fitting encoder on train_x_list[3] with onehot: ['fuelType', 'transmission', 'model'] and target-enc: ['fuelType', 'transmission', 'model']

[4] fitting encoder on train_x_list[4] with onehot: ['fuelType', 'transmission', 'model'] and target-enc: ['fuelType', 'transmission', 'model']

[5] fitting encoder on train_x_list[5] with onehot: ['fuelType', 'transmission', 'model'] and target-enc: ['fuelType', 'transmission', 'model']

[6] fitting encoder on train_x_list[6] with onehot: ['fuelType', 'transmission', 'model'] and target-enc: ['fuelType', 'transmiss

We replace our original lists with the encoded ones.

In [120]:
train_x_list = train_x_encoded_list
val_x_list   = val_x_encoded_list
test_x_list  = test_x_encoded_list

<a id="sec-4-filter"></a>
## 8. Feature Selection


For Feature Selection we first use filter methods and then use wrapper methods.

<a id="sec-4-filter"></a>
### 8.1 Filter Methods


We start with Filter methods, because they are computationally inexpensive and helpful for an initial screening of features. 

<a id="sec-4-1-variance"></a>
#### 8.1.1 Variance Threshold (Constant or Quasi-constant)


We therefore first calculate the variance, since values that are always the same dont help us with predicting.

In [121]:
# compute population variance, round for readability, sort by variance desc

train_x_list[0].var(ddof=0).round(6).sort_values(ascending=False)


mileage                        3.884743e+08
mileage_per_year               2.052373e+07
model                          6.994312e+06
transmission                   1.417398e+06
fuelType                       5.498129e+05
tax                            4.126654e+03
efficiency_ratio               1.476216e+02
mpg                            9.064972e+01
mpg_diff_transmission          8.576344e+01
year                           4.347687e+00
car_age                        4.347687e+00
previousOwners                 1.963678e+00
model_Focus                    2.465110e-01
fuelType_Petrol                2.181550e-01
fuelType_Diesel                2.177570e-01
model_Fiesta                   1.994510e-01
high_mileage_flag              1.875090e-01
engineSize                     1.757630e-01
transmission_Manual            1.160140e-01
very_high_mileage_flag         9.000600e-02
model_Kuga                     8.222100e-02
transmission_Automatic         7.530300e-02
transmission_Semi-Auto         4

We can see that some Features have a very low variance. We will set a threshold of 0.001 to remove features with variance below this value.

In [122]:
for i in range(len(train_x_list)):
    train_x_list[i], dropped_features = drop_low_variance_features(train_x_list[i], threshold=0.001)
    val_x_list[i] = val_x_list[i].drop(columns=dropped_features)
    test_x_list[i] = test_x_list[i].drop(columns=dropped_features)
    print(f"Dropped low-variance features: {list(dropped_features)}")

Dropped low-variance features: []
Dropped low-variance features: ['fuelType_Other', 'transmission_Other']
Dropped low-variance features: []
Dropped low-variance features: ['fuelType_Electric', 'fuelType_Other', 'transmission_Other']
Dropped low-variance features: ['fuelType_Electric']
Dropped low-variance features: []
Dropped low-variance features: []
Dropped low-variance features: []
Dropped low-variance features: ['transmission_Other']


<a id="sec-4-2-correlation"></a>
#### 8.1.1 Correlation Analysis
We decided us to not use Correlation Analysis, because it does not help our model.

<a id="sec-5-wrapper"></a>
### 8.2 Wrapper Method: Permutation-based Feature Selection


Instead of Recursive Feature Elimination (RFE), we use a permutation-importance
wrapper method with a Random Forest:

- For each brand, we train a `RandomForestRegressor` on all features of the
   training split.
- We compute permutation importance on the corresponding validation split
   (using R² as the scoring metric). This measures how much the model’s
   performance drops when a feature is randomly shuffled.
- We rank all features by their mean permutation importance and drop the
   bottom 15% (the least important ones). If the 0.15 Quantile is a negative value (bad for the model), we drop every feature that is negative or zero (more than 15%). If the 0.15 Quantile is bigger than 0.0001 it only drops values up to this threshold, so that we don't cut features that could help us.

In [123]:
%%time
Q = 0.15

# Create new lists so the original feature matrices remain unchanged
rf_train_x_list = [None] * len(train_x_list)
rf_val_x_list   = [None] * len(train_x_list)
rf_test_x_list  = [None] * len(train_x_list)

for i in range(len(train_x_list)):

    # Train a baseline RF on all features for this brand
    rf = RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=1,
        max_features="sqrt"
    )
    rf.fit(train_x_list[i], train_y_list[i])

    # Compute permutation importance on the validation set
    perm = permutation_importance(
        rf,
        val_x_list[i],
        val_y_list[i],
        n_repeats=20,
        random_state=42,
        scoring="r2",
        n_jobs=-1
    )

    imp_mean = pd.Series(
        perm.importances_mean,
        index=train_x_list[i].columns
    ).sort_values(ascending=False)

    q15 = imp_mean.quantile(Q)

    # Selection rule: prefer a small positive threshold when importance is noisy
    if q15 > 0.0001:
        threshold = 0.0001
        selected_feats = imp_mean[imp_mean > threshold].index.tolist()
        rule = "drop<=0.0001"
    elif q15 > 0:
        threshold = q15
        selected_feats = imp_mean[imp_mean >= threshold].index.tolist()
        rule = f"quantile@{Q:.2f}"
    else:
        threshold = 0.0
        selected_feats = imp_mean[imp_mean > threshold].index.tolist()
        rule = "drop<=0"

    print(
        f"\nBrand {i}: start={train_x_list[i].shape[1]}  "
        f"kept={len(selected_feats)}  dropped={train_x_list[i].shape[1] - len(selected_feats)}  "
        f"q15={q15:.6f}  rule={rule}  thr={threshold:.6f}"
    )

    # Store filtered matrices in separate lists
    rf_train_x_list[i] = train_x_list[i].loc[:, selected_feats].copy()
    rf_val_x_list[i]   = val_x_list[i].loc[:, selected_feats].copy()
    rf_test_x_list[i]  = test_x_list[i].loc[:, selected_feats].copy()

# Targets are unchanged, so references are fine
rf_train_y_list = train_y_list
rf_val_y_list   = val_y_list

PicklingError: Could not pickle the task to send it to the workers.

In [124]:
best_params_per_brand_et = [
    # Brand 0
    {'bootstrap': False, 'max_depth': 20, 'max_features': 0.5,
     'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 800},

    # Brand 1
    {'bootstrap': False, 'max_depth': 20, 'max_features': 0.5,
     'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 1200},

    # Brand 2
    {'bootstrap': False, 'max_depth': 20, 'max_features': 0.5,
     'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 800},

    # Brand 3
    {'bootstrap': False, 'max_depth': 20, 'max_features': 'sqrt',
     'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 1200},

    # Brand 4
    {'bootstrap': False, 'max_depth': None, 'max_features': 0.5,
     'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 1200},

    # Brand 5
    {'bootstrap': False, 'max_depth': 20, 'max_features': 0.5,
     'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 1200},

    # Brand 6
    {'bootstrap': False, 'max_depth': 15, 'max_features': 0.5,
     'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 500},

    # Brand 7
    {'bootstrap': False, 'max_depth': 20, 'max_features': 0.5,
     'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 1200},

    # Brand 8
    {'bootstrap': False, 'max_depth': 15, 'max_features': 0.5,
     'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 1200}]

Q = 0.15
selected_features = []

for i in range(len(train_x_list)):

    # 1) train on ALL features (ExtraTrees with your tuned params)
    et = ExtraTreesRegressor(
        random_state=42,
        n_jobs=-1,
        **best_params_per_brand_et[i]
    )
    et.fit(train_x_list[i], train_y_list[i])

    # 2) permutation importance on val
    perm = permutation_importance(
        et,
        val_x_list[i],
        val_y_list[i],
        n_repeats=20,
        random_state=42,
        scoring="r2",
        n_jobs=1
    )

    imp_mean = pd.Series(
        perm.importances_mean,
        index=train_x_list[i].columns
    ).sort_values(ascending=False)

    # 3) decision rule:
    # - if q15 > 0: do quantile cut (like before)
    # - else: drop everything with importance <= 0
    q15 = imp_mean.quantile(Q)

    if q15 > 0.0001:
        threshold = 0.0001
        selected_feats = imp_mean[imp_mean > threshold].index.tolist()
        dropped_feats  = imp_mean[imp_mean <= threshold].index.tolist()
        rule = "drop<=0.0001"
    elif q15 > 0:
        threshold = q15
        selected_feats = imp_mean[imp_mean >= threshold].index.tolist()
        dropped_feats  = imp_mean[imp_mean <  threshold].index.tolist()
        rule = f"quantile@{Q:.2f}"
    else:
        threshold = 0.0
        selected_feats = imp_mean[imp_mean > threshold].index.tolist()
        dropped_feats  = imp_mean[imp_mean <= threshold].index.tolist()
        rule = "drop<=0"

    print(f"\nBrand {i}: start={train_x_list[i].shape[1]}  "
          f"kept={len(selected_feats)}  dropped={len(dropped_feats)}  "
          f"q15={q15:.6f}  rule={rule}  thr={threshold:.6f}")

    # update
    selected_features.append(selected_feats)
    train_x_list[i] = train_x_list[i][selected_feats]
    val_x_list[i]   = val_x_list[i][selected_feats]
    test_x_list[i]  = test_x_list[i][selected_feats]


Brand 0: start=40  kept=34  dropped=6  q15=0.000039  rule=quantile@0.15  thr=0.000039

Brand 1: start=41  kept=36  dropped=5  q15=0.000126  rule=drop<=0.0001  thr=0.000100

Brand 2: start=44  kept=35  dropped=9  q15=-0.000010  rule=drop<=0  thr=0.000000

Brand 3: start=36  kept=34  dropped=2  q15=0.000485  rule=drop<=0.0001  thr=0.000100

Brand 4: start=42  kept=35  dropped=7  q15=0.000081  rule=quantile@0.15  thr=0.000081

Brand 5: start=38  kept=32  dropped=6  q15=0.000080  rule=quantile@0.15  thr=0.000080

Brand 6: start=36  kept=30  dropped=6  q15=0.000025  rule=quantile@0.15  thr=0.000025

Brand 7: start=35  kept=29  dropped=6  q15=0.000045  rule=quantile@0.15  thr=0.000045

Brand 8: start=34  kept=29  dropped=5  q15=0.000113  rule=drop<=0.0001  thr=0.000100


In [125]:
# help to not always have to run again
#for i in range(len(train_x_list)):
#    print(selected_features[i])
#    train_x_list[i] = train_x_list[i][selected_features[i]]
#    val_x_list[i]   = val_x_list[i][selected_features[i]]
#    test_x_list[i]  = test_x_list[i][selected_features[i]]

## 9. Models

For models we tried Random Forest Regressor, Extra Tree Regressor and Hist Gradient Boosting Regressor. For each of those we use grid search and hardcode the result to be able to faster test the models.

### 9.1 Random Forest Regressor

First we use grid search for Random Forest.

In [34]:
param_grid = {
    "n_estimators": [200, 300, 400, 500, 600],
    "max_depth": [15, 17, 20, 23, 25],
    "max_features": ["sqrt", 0.5],
    "min_samples_split": [3, 5, 7],
    "min_samples_leaf": [1, 2],
}

grid = list(ParameterGrid(param_grid))
n_total = len(grid)

best_params_per_brand_rf = []
best_score_per_brand_rf = []

for b in range(len(train_x_list)):
    Xtr = rf_train_x_list[b]
    ytr = rf_train_y_list[b]
    Xva = val_x_list[b]
    yva = val_y_list[b]

    best_mae = float("inf")
    best_params = None

    print("\n==============================")
    print(f"Brand {b}: grid search over {n_total} configs")

    for i, params in enumerate(grid, start=1):
        print(f"[{i}/{n_total}] params: {params}")

        model = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
        model.fit(Xtr, ytr)

        pred = model.predict(Xva)
        mae = mean_absolute_error(yva, pred)

        print(f"    -> MAE: {mae:.3f}")

        if mae < best_mae:
            best_mae = mae
            best_params = params
            print(f"    NEW BEST: MAE={best_mae:.3f} | params={best_params}")

    best_params_per_brand_rf.append(best_params)
    best_score_per_brand_rf.append(best_mae)

print("\n===== FINAL BEST PER BRAND =====")
for b, (p, s) in enumerate(zip(best_params_per_brand_rf, best_score_per_brand_rf)):
    print(f"Brand {b}: MAE={s:.3f} | params={p}")

    -> MAE: 882.629
[45/300] params: {'max_depth': 15, 'max_features': 0.5, 'min_samples_leaf': 1, 'min_samples_split': 7, 'n_estimators': 600}
    -> MAE: 882.748
[46/300] params: {'max_depth': 15, 'max_features': 0.5, 'min_samples_leaf': 2, 'min_samples_split': 3, 'n_estimators': 200}
    -> MAE: 886.020
[47/300] params: {'max_depth': 15, 'max_features': 0.5, 'min_samples_leaf': 2, 'min_samples_split': 3, 'n_estimators': 300}
    -> MAE: 885.787
[48/300] params: {'max_depth': 15, 'max_features': 0.5, 'min_samples_leaf': 2, 'min_samples_split': 3, 'n_estimators': 400}
    -> MAE: 885.957
[49/300] params: {'max_depth': 15, 'max_features': 0.5, 'min_samples_leaf': 2, 'min_samples_split': 3, 'n_estimators': 500}
    -> MAE: 886.093
[50/300] params: {'max_depth': 15, 'max_features': 0.5, 'min_samples_leaf': 2, 'min_samples_split': 3, 'n_estimators': 600}
    -> MAE: 886.120
[51/300] params: {'max_depth': 15, 'max_features': 0.5, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators

Help: hardcode of the result to calculate quicker

In [89]:
best_params_per_brand_rf = [
    # Brand 0 (MAE=880.558)
    {'max_depth': 17, 'max_features': 0.5,
     'min_samples_leaf': 1, 'min_samples_split': 7, 'n_estimators': 200},

    # Brand 1 (MAE=1742.622)
    {'max_depth': 23, 'max_features': 'sqrt',
     'min_samples_leaf': 1, 'min_samples_split': 3, 'n_estimators': 200},

    # Brand 2 (MAE=1135.238)
    {'max_depth': 17, 'max_features': 'sqrt',
     'min_samples_leaf': 1, 'min_samples_split': 3, 'n_estimators': 600},

    # Brand 3 (MAE=714.560)
    {'max_depth': 15, 'max_features': 0.5,
     'min_samples_leaf': 1, 'min_samples_split': 3, 'n_estimators': 600},

    # Brand 4 (MAE=1617.368)
    {'max_depth': 25, 'max_features': 'sqrt',
     'min_samples_leaf': 1, 'min_samples_split': 3, 'n_estimators': 600},

    # Brand 5 (MAE=1665.284)
    {'max_depth': 15, 'max_features': 0.5,
     'min_samples_leaf': 1, 'min_samples_split': 3, 'n_estimators': 300},

    # Brand 6 (MAE=810.374)
    {'max_depth': 15, 'max_features': 0.5,
     'min_samples_leaf': 2, 'min_samples_split': 7, 'n_estimators': 400},

    # Brand 7 (MAE=996.433)
    {'max_depth': 15, 'max_features': 'sqrt',
     'min_samples_leaf': 1, 'min_samples_split': 3, 'n_estimators': 300},

    # Brand 8 (MAE=825.807)
    {'max_depth': 15, 'max_features': 'sqrt',
     'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 400},
]

Then we use the found parameters for the algorithm to predict the prices of our test data. We therefore also log transform the target.

In [90]:
metrics_per_brand = []
test_pred_list_rf = []
test_pred_list_rf_log = []

for i in range(len(rf_train_x_list)):

    y_train_i = train_y_list[i]
    y_val_i   = val_y_list[i]

    if isinstance(y_train_i, pd.DataFrame):
        y_train_i = y_train_i.iloc[:, 0]
    if isinstance(y_val_i, pd.DataFrame):
        y_val_i = y_val_i.iloc[:, 0]

    # train (log)
    y_train_log = np.log1p(y_train_i)

    model_i = RandomForestRegressor(
        **best_params_per_brand_rf[i],
        random_state=42,
        n_jobs=-1
    )
    model_i.fit(rf_train_x_list[i], y_train_log)

    # val pred (back to original)
    pred_val_log = model_i.predict(rf_val_x_list[i])
    pred_val = np.expm1(pred_val_log)

    mae   = mean_absolute_error(y_val_i, pred_val)
    rmse  = np.sqrt(mean_squared_error(y_val_i, pred_val))
    r2    = r2_score(y_val_i, pred_val)
    medae = median_absolute_error(y_val_i, pred_val)

    metrics_per_brand.append({
        "brand_idx": i,   # oder "brand": brand_names[i]
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "medae": medae,
    })

    # final: train+val -> test prediction
    x_trainval_i = pd.concat([rf_train_x_list[i], rf_val_x_list[i]], axis=0)
    y_trainval_i = pd.concat([y_train_i, y_val_i], axis=0)

    model_final = RandomForestRegressor(
        **best_params_per_brand_rf[i],
        random_state=42,
        n_jobs=-1
    )
    model_final.fit(x_trainval_i, np.log1p(y_trainval_i))

    pred_test_log = model_final.predict(rf_test_x_list[i])
    pred_test = np.expm1(pred_test_log)

    test_pred_list_rf.append(pd.Series(pred_test, index=rf_test_x_list[i].index, name="price"))
    test_pred_list_rf_log.append(pd.Series(pred_test_log, index=rf_test_x_list[i].index, name="price_log"))


metrics_rf = pd.DataFrame(metrics_per_brand)

ValueError: Expected 2D array, got scalar array instead:
array=nan.
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

After that we build the submission.

In [ ]:
# put all brand predictions back together
test_pred_all = pd.concat(test_pred_list_rf).sort_index()

# build submission
submission = test_pred_all.reset_index()
submission.columns = ["CarID", "price"]
submission.to_csv("submission.csv", index=False)

print(submission.head())
print("total preds:", len(submission))

After that we upload the submission in the submission folder, by the suffix _rf we know which model it was.

In [ ]:
# Save
sub_dir = os.path.join(data_dir, "submissions")
os.makedirs(sub_dir, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M")
sub_path = os.path.join(sub_dir, f"30_submission_{ts}_rf.csv")
submission.to_csv(sub_path, index=False)

print(f"Submission saved to: {sub_path}")
display(submission.head(10))


## 9.2 Extra Trees Regressor

For Extra Trees we also use grid search first.

In [43]:
param_grid_et = {
    "n_estimators": [500, 800, 1200],
    "max_depth": [None, 10, 15, 20],
    "max_features": ["sqrt", 0.5],
    "min_samples_split": [2, 3, 5],
    "min_samples_leaf": [1, 2, 3, 5],
    "bootstrap": [False],
}

grid = list(ParameterGrid(param_grid_et))
n_total = len(grid)

best_params_per_brand_et = []
best_score_per_brand_et = []

for b in range(len(train_x_list)):
    Xtr = train_x_list[b]
    ytr = train_y_list[b]
    Xva = val_x_list[b]
    yva = val_y_list[b]

    # falls y DataFrame -> Series
    if isinstance(ytr, pd.DataFrame): ytr = ytr.iloc[:, 0]
    if isinstance(yva, pd.DataFrame): yva = yva.iloc[:, 0]

    best_mae = float("inf")
    best_params = None

    print("\n==============================")
    print(f"Brand {b}: ExtraTrees grid search over {n_total} configs")

    for i, params in enumerate(grid, start=1):
        print(f"[{i}/{n_total}] params: {params}")

        model = ExtraTreesRegressor(**params, random_state=42, n_jobs=-1)

        # ---- log1p training ----
        model.fit(Xtr, np.log1p(ytr))

        # ---- predict + inverse ----
        pred_log = model.predict(Xva)
        pred = np.expm1(pred_log)

        mae = mean_absolute_error(yva, pred)
        print(f"    -> MAE (log1p): {mae:.3f}")

        if mae < best_mae:
            best_mae = mae
            best_params = params
            print(f"    NEW BEST: MAE={best_mae:.3f} | params={best_params}")

    best_params_per_brand_et.append(best_params)
    best_score_per_brand_et.append(best_mae)

print("\n===== FINAL BEST EXTRA-TREES PER BRAND =====")
for b, (p, s) in enumerate(zip(best_params_per_brand_et, best_score_per_brand_et)):
    print(f"Brand {b}: MAE={s:.3f} | params={p}")


Brand 0: ExtraTrees grid search over 288 configs
[1/288] params: {'bootstrap': False, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 500}
    -> MAE (log1p): 945.193
    NEW BEST: MAE=945.193 | params={'bootstrap': False, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 500}
[2/288] params: {'bootstrap': False, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 800}
    -> MAE (log1p): 945.078
    NEW BEST: MAE=945.078 | params={'bootstrap': False, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 800}
[3/288] params: {'bootstrap': False, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 1200}
    -> MAE (log1p): 945.169
[4/288] params: {'bootstrap': False, 'max_depth': None, 'max_features': 'sqrt', 'min_sa

Help: hardcode of the result to calculate quicker

In [ ]:
best_params_per_brand_et = [
    # Brand 0 (MAE=876.688)
    {'bootstrap': False, 'max_depth': 20, 'max_features': 0.5,
     'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 800},

    # Brand 1 (MAE=1715.926)
    {'bootstrap': False, 'max_depth': 20, 'max_features': 0.5,
     'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 500},

    # Brand 2 (MAE=1122.832)
    {'bootstrap': False, 'max_depth': 20, 'max_features': 0.5,
     'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 1200},

    # Brand 3 (MAE=727.604)
    {'bootstrap': False, 'max_depth': 15, 'max_features': 0.5,
     'min_samples_leaf': 1, 'min_samples_split': 3, 'n_estimators': 1200},

    # Brand 4 (MAE=1574.483)
    {'bootstrap': False, 'max_depth': None, 'max_features': 'sqrt',
     'min_samples_leaf': 1, 'min_samples_split': 3, 'n_estimators': 1200},

    # Brand 5 (MAE=1658.852)
    {'bootstrap': False, 'max_depth': 20, 'max_features': 0.5,
     'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 500},

    # Brand 6 (MAE=811.462)
    {'bootstrap': False, 'max_depth': None, 'max_features': 0.5,
     'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 500},

    # Brand 7 (MAE=1018.251)
    {'bootstrap': False, 'max_depth': 15, 'max_features': 0.5,
     'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 500},

    # Brand 8 (MAE=791.526)
    {'bootstrap': False, 'max_depth': None, 'max_features': 'sqrt',
     'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 800},
]

Then we use the found parameters for the algorithm to predict the prices of our test data. We therefore also log transform the target.

In [ ]:
test_pred_list_et = []
test_pred_list_et_log = []
metrics_per_brand_et = []

val_pred_list_et_log = []

for i in range(len(train_x_list)):
    X_tr, X_va, X_te = train_x_list[i], val_x_list[i], test_x_list[i]
    y_tr, y_va = train_y_list[i], val_y_list[i]
    if isinstance(y_tr, pd.DataFrame): y_tr = y_tr.iloc[:,0]
    if isinstance(y_va, pd.DataFrame): y_va = y_va.iloc[:,0]

    params_et = best_params_per_brand_et[i].copy()
    params_et['criterion'] = 'absolute_error'

    # train -> val preds
    et_val = ExtraTreesRegressor(**params_et, random_state=42, n_jobs=-1)
    et_val.fit(X_tr, np.log1p(y_tr))

    pred_va_log = et_val.predict(X_va)
    pred_va = np.expm1(pred_va_log)

    val_pred_list_et_log.append(pd.Series(pred_va_log, index=X_va.index, name="price_log"))

    # metrics
    mae   = mean_absolute_error(y_va, pred_va)
    rmse  = np.sqrt(mean_squared_error(y_va, pred_va))
    r2    = r2_score(y_va, pred_va)
    medae = median_absolute_error(y_va, pred_va)

    metrics_per_brand_et.append({"brand_idx": i, "mae": mae, "rmse": rmse, "r2": r2, "medae": medae})

    # final: train+val -> test
    x_trainval = pd.concat([X_tr, X_va], axis=0)
    y_trainval = pd.concat([y_tr, y_va], axis=0)

    et_final = ExtraTreesRegressor(**params_et, random_state=42, n_jobs=-1)
    et_final.fit(x_trainval, np.log1p(y_trainval))

    pred_log_te = et_final.predict(X_te)
    pred_te = np.expm1(pred_log_te)

    test_pred_list_et.append(pd.Series(pred_te, index=X_te.index, name="price"))
    test_pred_list_et_log.append(pd.Series(pred_log_te, index=X_te.index, name="price_log"))
metrics_et = pd.DataFrame(metrics_per_brand_et)

After that we build the submission.

In [ ]:
# put all brand predictions back together
test_pred_all = pd.concat(test_pred_list_et).sort_index()

# build submission
submission = test_pred_all.reset_index()
submission.columns = ["CarID", "price"]
submission.to_csv("submission.csv", index=False)

print(submission.head())
print("total preds:", len(submission))

After that we upload the submission in the submission folder, by the suffix _etr we know which model it was.

In [ ]:
# Save
sub_dir = os.path.join(data_dir, "submissions")
os.makedirs(sub_dir, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M")
sub_path = os.path.join(sub_dir, f"30_submission_{ts}_etr.csv")
submission.to_csv(sub_path, index=False)

print(f"Submission saved to: {sub_path}")
display(submission.head(10))


## 9.3 Hist Gradient Boosting Regressor

In [47]:
# ---- kleines, "high impact" Grid für HGB ----
param_grid_hgb = {
    "max_depth": [6, 8],
    "learning_rate": [0.03, 0.06, 0.1],
    "min_samples_leaf": [10, 30, 50],
    "l2_regularization": [0.02, 0.05, 0.1],
    "max_bins": [128, 255],
    "max_iter": [4000],
    "early_stopping": [True],
}

grid = list(ParameterGrid(param_grid_hgb))
n_total = len(grid)

best_params_per_brand_hgb = []
best_score_per_brand_hgb  = []

for b in range(len(train_x_list)):
    Xtr = train_x_list[b]
    ytr = train_y_list[b]
    Xva = val_x_list[b]
    yva = val_y_list[b]

    # falls y DataFrame -> Series
    if isinstance(ytr, pd.DataFrame): ytr = ytr.iloc[:, 0]
    if isinstance(yva, pd.DataFrame): yva = yva.iloc[:, 0]

    best_mae = float("inf")
    best_params = None

    print("\n==============================")
    print(f"Brand {b}: HGB grid search over {n_total} configs")

    for i, params in enumerate(grid, start=1):
        print(f"[{i}/{n_total}] params: {params}")

        model = HistGradientBoostingRegressor(
            **params,
            random_state=42
        )

        # ---- log1p training ----
        model.fit(Xtr, np.log1p(ytr))

        # ---- predict + inverse ----
        pred_log = model.predict(Xva)
        pred = np.expm1(pred_log)

        mae = mean_absolute_error(yva, pred)
        print(f"    -> MAE (log1p): {mae:.3f}")

        if mae < best_mae:
            best_mae = mae
            best_params = params
            print(f"    NEW BEST: MAE={best_mae:.3f} | params={best_params}")

    best_params_per_brand_hgb.append(best_params)
    best_score_per_brand_hgb.append(best_mae)

print("\n===== FINAL BEST HGB PER BRAND =====")
for b, (p, s) in enumerate(zip(best_params_per_brand_hgb, best_score_per_brand_hgb)):
    print(f"Brand {b}: MAE={s:.3f} | params={p}")


Brand 0: HGB grid search over 108 configs
[1/108] params: {'early_stopping': True, 'l2_regularization': 0.02, 'learning_rate': 0.03, 'max_bins': 128, 'max_depth': 6, 'max_iter': 4000, 'min_samples_leaf': 10}
    -> MAE (log1p): 891.662
    NEW BEST: MAE=891.662 | params={'early_stopping': True, 'l2_regularization': 0.02, 'learning_rate': 0.03, 'max_bins': 128, 'max_depth': 6, 'max_iter': 4000, 'min_samples_leaf': 10}
[2/108] params: {'early_stopping': True, 'l2_regularization': 0.02, 'learning_rate': 0.03, 'max_bins': 128, 'max_depth': 6, 'max_iter': 4000, 'min_samples_leaf': 30}
    -> MAE (log1p): 896.456
[3/108] params: {'early_stopping': True, 'l2_regularization': 0.02, 'learning_rate': 0.03, 'max_bins': 128, 'max_depth': 6, 'max_iter': 4000, 'min_samples_leaf': 50}
    -> MAE (log1p): 916.562
[4/108] params: {'early_stopping': True, 'l2_regularization': 0.02, 'learning_rate': 0.03, 'max_bins': 128, 'max_depth': 8, 'max_iter': 4000, 'min_samples_leaf': 10}
    -> MAE (log1p): 894.

Help: hardcode of the result to calculate quicker

In [ ]:
best_params_per_brand_hgb = [
    # Brand 0
    {'early_stopping': True, 'l2_regularization': 0.05, 'learning_rate': 0.06,
     'max_bins': 255, 'max_depth': 8, 'max_iter': 4000, 'min_samples_leaf': 30,
     'n_iter_no_change': 50, 'validation_fraction': 0.1},

    # Brand 1
    {'early_stopping': True, 'l2_regularization': 0.05, 'learning_rate': 0.06,
     'max_bins': 255, 'max_depth': 8, 'max_iter': 4000, 'min_samples_leaf': 30,
     'n_iter_no_change': 50, 'validation_fraction': 0.1},

    # Brand 2
    {'early_stopping': True, 'l2_regularization': 0.05, 'learning_rate': 0.06,
     'max_bins': 255, 'max_depth': 8, 'max_iter': 4000, 'min_samples_leaf': 30,
     'n_iter_no_change': 50, 'validation_fraction': 0.1},

    # Brand 3
    {'early_stopping': True, 'l2_regularization': 0.05, 'learning_rate': 0.06,
     'max_bins': 255, 'max_depth': 6, 'max_iter': 4000, 'min_samples_leaf': 30,
     'n_iter_no_change': 50, 'validation_fraction': 0.1},

    # Brand 4
    {'early_stopping': True, 'l2_regularization': 0.05, 'learning_rate': 0.06,
     'max_bins': 255, 'max_depth': 6, 'max_iter': 4000, 'min_samples_leaf': 50,
     'n_iter_no_change': 50, 'validation_fraction': 0.1},

    # Brand 5
    {'early_stopping': True, 'l2_regularization': 0.05, 'learning_rate': 0.06,
     'max_bins': 255, 'max_depth': 8, 'max_iter': 4000, 'min_samples_leaf': 30,
     'n_iter_no_change': 50, 'validation_fraction': 0.1},

    # Brand 6
    {'early_stopping': True, 'l2_regularization': 0.05, 'learning_rate': 0.06,
     'max_bins': 255, 'max_depth': 8, 'max_iter': 4000, 'min_samples_leaf': 30,
     'n_iter_no_change': 50, 'validation_fraction': 0.1},

    # Brand 7
    {'early_stopping': True, 'l2_regularization': 0.05, 'learning_rate': 0.06,
     'max_bins': 255, 'max_depth': 6, 'max_iter': 4000, 'min_samples_leaf': 30,
     'n_iter_no_change': 50, 'validation_fraction': 0.1},

    # Brand 8
    {'early_stopping': True, 'l2_regularization': 0.05, 'learning_rate': 0.06,
     'max_bins': 255, 'max_depth': 8, 'max_iter': 4000, 'min_samples_leaf': 30,
     'n_iter_no_change': 50, 'validation_fraction': 0.1},
]

Then we use the found parameters for the algorithm to predict the prices of our test data. We therefore also log transform the target.

In [ ]:
test_pred_list_hgb = []
test_pred_list_hgb_log = []
metrics_per_brand_hgb = []

val_pred_list_hgb_log = []
val_pred_list_hgb = []

for i in range(len(train_x_list)):

    y_tr = train_y_list[i]
    y_va = val_y_list[i]
    if isinstance(y_tr, pd.DataFrame): y_tr = y_tr.iloc[:, 0]
    if isinstance(y_va, pd.DataFrame): y_va = y_va.iloc[:, 0]

    # val fit + preds
    params_hgb = best_params_per_brand_hgb[i].copy()
    params_hgb['loss'] = 'absolute_error'

    hgb = HistGradientBoostingRegressor(**params_hgb, random_state=42)
    hgb.fit(train_x_list[i], np.log1p(y_tr))

    pred_va_log = hgb.predict(val_x_list[i])
    pred_va = np.expm1(pred_va_log)

    val_pred_list_hgb_log.append(pd.Series(pred_va_log, index=val_x_list[i].index, name="price_log"))
    val_pred_list_hgb.append(pd.Series(pred_va, index=val_x_list[i].index, name="price"))  # optional

    mae   = mean_absolute_error(y_va, pred_va)
    rmse  = np.sqrt(mean_squared_error(y_va, pred_va))
    r2    = r2_score(y_va, pred_va)
    medae = median_absolute_error(y_va, pred_va)

    metrics_per_brand_hgb.append({"brand_idx": i, "mae": mae, "rmse": rmse, "r2": r2, "medae": medae})

    print(f"Brand {i}: val MAE={mae:.3f}, RMSE={rmse:.3f}, R2={r2:.3f}, MedAE={medae:.3f}")

    # final train+val -> test
    x_trainval = pd.concat([train_x_list[i], val_x_list[i]], axis=0)
    y_trainval = pd.concat([y_tr, y_va], axis=0)

    params_hgb_final = best_params_per_brand_hgb[i].copy()
    params_hgb_final['loss'] = 'absolute_error'

    hgb_final = HistGradientBoostingRegressor(**params_hgb_final, random_state=42)
    hgb_final.fit(x_trainval, np.log1p(y_trainval))

    pred_te_log = hgb_final.predict(test_x_list[i])
    pred_te = np.expm1(pred_te_log)

    test_pred_list_hgb.append(pd.Series(pred_te, index=test_x_list[i].index, name="price"))
    test_pred_list_hgb_log.append(pd.Series(pred_te_log, index=test_x_list[i].index, name="price_log"))
metrics_hgb = pd.DataFrame(metrics_per_brand_hgb)

After that we build the submission.

In [ ]:
# put all brand predictions back together
test_pred_list_hgb = pd.concat(test_pred_list_hgb).sort_index()

# build submission
submission = test_pred_list_hgb.reset_index()
submission.columns = ["CarID", "price"]
submission.to_csv("submission.csv", index=False)

print(submission.head())
print("total preds:", len(submission))

After that we upload the submission in the submission folder, by the suffix _hgbr we know which model it was.

In [ ]:
# Save
sub_dir = os.path.join(data_dir, "submissions")
os.makedirs(sub_dir, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M")
sub_path = os.path.join(sub_dir, f"30_submission_{ts}_hgbr.csv")
submission.to_csv(sub_path, index=False)

print(f"Submission saved to: {sub_path}")
display(submission.head(10))


## 9.4 Hybrid Model (Extra Tree Regressor * 0,6 + High Gradient Boosting * 0,4)

For the hybrid model we take the predictions of both models, apply their wheigts and add them toegther. We then create the submission.

In [ ]:
w = 0.6
metrics_per_brand_blend = []

for i in range(len(val_y_list)):
    y_va = val_y_list[i]
    if isinstance(y_va, pd.DataFrame): y_va = y_va.iloc[:, 0]

    pred_et_va_log  = val_pred_list_et_log[i].copy()
    pred_hgb_va_log = val_pred_list_hgb_log[i].copy()

    # align indices
    y_va_s = pd.Series(y_va.values, index=y_va.index)
    y_va_s.index = y_va_s.index.astype(int)

    pred_et_va_log.index  = pred_et_va_log.index.astype(int)
    pred_hgb_va_log.index = pred_hgb_va_log.index.astype(int)

    y_va_s = y_va_s.sort_index()
    pred_et_va_log  = pred_et_va_log.sort_index()
    pred_hgb_va_log = pred_hgb_va_log.sort_index()

    common_idx = y_va_s.index.intersection(pred_et_va_log.index).intersection(pred_hgb_va_log.index)

    # blend in log-space -> back to original
    pred_blend_va = np.expm1(
        w * pred_et_va_log.loc[common_idx] + (1 - w) * pred_hgb_va_log.loc[common_idx]
    )

    y_true = y_va_s.loc[common_idx]

    mae   = mean_absolute_error(y_true, pred_blend_va)
    rmse  = np.sqrt(mean_squared_error(y_true, pred_blend_va))
    r2    = r2_score(y_true, pred_blend_va)
    medae = median_absolute_error(y_true, pred_blend_va)

    metrics_per_brand_blend.append({
        "brand_idx": i,
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "medae": medae,
    })

metrics_blend = pd.DataFrame(metrics_per_brand_blend)

In [ ]:
test_pred_list_final = []

w = 0.6  # ET-Gewicht in log-space, HGB = 1-w

for i in range(len(test_pred_list_et_log)):
    pred_et_log  = test_pred_list_et_log[i].copy()
    pred_hgb_log = test_pred_list_hgb_log[i].copy()

    # Indizes alignen
    pred_et_log.index  = pred_et_log.index.astype(int)
    pred_hgb_log.index = pred_hgb_log.index.astype(int)

    pred_et_log  = pred_et_log.sort_index()
    pred_hgb_log = pred_hgb_log.sort_index()

    common_idx = pred_et_log.index.intersection(pred_hgb_log.index)

    # log-space blend -> zurück in Originalskala
    pred_final = np.expm1(
        w * pred_et_log.loc[common_idx] + (1 - w) * pred_hgb_log.loc[common_idx]
    )

    test_pred_list_final.append(pd.Series(pred_final, index=common_idx))

test_pred_all = pd.concat(test_pred_list_final).sort_index()

submission = test_pred_all.reset_index()
submission.columns = ["CarID", "price"]

print(submission.head())
print("total preds:", len(submission))
print("n_nans:", test_pred_all.isna().sum())

After that we upload the submission in the submission folder, by the suffix _hybrid we know which model it was.

In [ ]:
# Save
sub_dir = os.path.join(data_dir, "submissions")
os.makedirs(sub_dir, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M")
sub_path = os.path.join(sub_dir, f"30_submission_{ts}_hybrid.csv")
submission.to_csv(sub_path, index=False)

print(f"Submission saved to: {sub_path}")
display(submission.head(10))

# 10. Evaluation

First we check if our models overfit with Random Forest as example.

In [ ]:
test_pred_list_et = []
train_mae_list_et = []
val_mae_list_et = []

train_weights = []  
val_weights = []   

for i in range(len(train_x_list)):
    X_tr = train_x_list[i]
    X_va = val_x_list[i]
    X_te = test_x_list[i]

    y_tr = train_y_list[i]
    y_va = val_y_list[i]

    # falls y DataFrame -> Series
    if isinstance(y_tr, pd.DataFrame): y_tr = y_tr.iloc[:, 0]
    if isinstance(y_va, pd.DataFrame): y_va = y_va.iloc[:, 0]

    # weights = amount of samples
    train_weights.append(len(y_tr))
    val_weights.append(len(y_va))

    # ====== 1) TRAIN fit -> train + val MAE ======
    et_val = ExtraTreesRegressor(
        **best_params_per_brand_et[i],
        random_state=42,
        n_jobs=-1
    )

    et_val.fit(X_tr, np.log1p(y_tr))

    # train MAE
    pred_tr = np.expm1(et_val.predict(X_tr))
    mae_tr = mean_absolute_error(y_tr, pred_tr)
    train_mae_list_et.append(mae_tr)

    # val MAE
    pred_va = np.expm1(et_val.predict(X_va))
    mae_va = mean_absolute_error(y_va, pred_va)
    val_mae_list_et.append(mae_va)

    print(f"Brand {i}: train MAE = {mae_tr:.3f} | val MAE = {mae_va:.3f}")

    # ====== 2) FINAL fit (train+val) -> test preds ======
    x_trainval = pd.concat([X_tr, X_va], axis=0)
    y_trainval = pd.concat([y_tr, y_va], axis=0)

    et_final = ExtraTreesRegressor(
        **best_params_per_brand_et[i],
        random_state=42,
        n_jobs=-1
    )
    et_final.fit(x_trainval, np.log1p(y_trainval))
    pred_te = np.expm1(et_final.predict(X_te))

    test_pred_list_et.append(pd.Series(pred_te, index=X_te.index, name="price"))

# ===== weighted averages =====
train_weights = np.array(train_weights, dtype=float)
val_weights   = np.array(val_weights, dtype=float)

weighted_train_mae = np.sum(train_weights * np.array(train_mae_list_et)) / np.sum(train_weights)
weighted_val_mae   = np.sum(val_weights   * np.array(val_mae_list_et))   / np.sum(val_weights)

print("\n=== AVERAGES ===")
print(f"Avg train MAE (weighted):    {weighted_train_mae:.3f}")
print(f"Avg val MAE   (weighted):    {weighted_val_mae:.3f}")

The train–validation gap shows clear overfitting: the weighted train MAE is 562.532 vs. weighted val MAE is 1149.61. 
The validation MAE is about 104.36% higher than the training MAE.
We accept this because this notebook’s models still achieved the best test performance in the project.

After that we want to compare the different models by mae, rmse, r2 and medea. MAE reflects the average error level, RMSE shows how strongly the model is affected by large errors, R² indicates how well the model captures the overall structure of the data, and MedAE is like MAE without being influenced by outliers.

In [ ]:
weights = pd.Series(
    {i: len(val_x_list[i]) for i in range(len(val_x_list))},
    name="weight"
)
def add_weighted_row_metric(df_metric, weights, label="WEIGHTED"):
    df2 = df_metric.copy()
    w = df2["brand_idx"].map(weights).astype(float)

    metric_cols = [c for c in df2.columns if c != "brand_idx"]
    weighted = {"brand_idx": label}
    for c in metric_cols:
        s = df2[c].astype(float)
        mask = s.notna() & w.notna() & (w > 0)
        weighted[c] = (s[mask] * w[mask]).sum() / w[mask].sum()

    return pd.concat([df2, pd.DataFrame([weighted])], ignore_index=True)

In [ ]:
def metric_table(compare_df, metric, weights=None, add_weighted=True):
    dfm = compare_df[pd.to_numeric(compare_df["brand_idx"], errors="coerce").notna()].copy()
    dfm["brand_idx"] = dfm["brand_idx"].astype(int)
    dfm = dfm.sort_values("brand_idx")

    out = dfm[["brand_idx", f"et_{metric}", f"rf_{metric}", f"hgb_{metric}", f"blend_{metric}"]].copy()

    if add_weighted and weights is not None:
        out = add_weighted_row_metric(out, weights, label="WEIGHTED")

    return out
metrics_et_cmp = metrics_et.rename(columns={
    "mae": "et_mae", "rmse": "et_rmse", "r2": "et_r2", "medae": "et_medae"
})
metrics_rf_cmp = metrics_rf.rename(columns={
    "mae": "rf_mae", "rmse": "rf_rmse", "r2": "rf_r2", "medae": "rf_medae"
})
metrics_hgb_cmp = metrics_hgb.rename(columns={
    "mae": "hgb_mae", "rmse": "hgb_rmse", "r2": "hgb_r2", "medae": "hgb_medae"
})
metrics_blend_cmp = metrics_blend.rename(columns={
    "mae": "blend_mae", "rmse": "blend_rmse", "r2": "blend_r2", "medae": "blend_medae"
})

# --- merge zu einer Vergleichstabelle ---
compare_df = (
    metrics_et_cmp.merge(metrics_rf_cmp, on="brand_idx", how="outer")
                 .merge(metrics_hgb_cmp, on="brand_idx", how="outer")
                 .merge(metrics_blend_cmp, on="brand_idx", how="outer")
                 .sort_values("brand_idx")
                 .reset_index(drop=True)
)

mae_table   = metric_table(compare_df, "mae",   weights)
rmse_table  = metric_table(compare_df, "rmse",  weights)
r2_table    = metric_table(compare_df, "r2",    weights)
medae_table = metric_table(compare_df, "medae", weights)

brand_map = {
    0: "Ford",
    1: "Mercedes-Benz",
    2: "Volkswagen",
    3: "Opel",
    4: "BMW",
    5: "Audi",
    6: "Toyota",
    7: "Škoda",
    8: "Hyundai",
}

for tbl in [mae_table, rmse_table, r2_table, medae_table]:
    tbl["brand_idx"] = tbl["brand_idx"].map(brand_map).fillna(tbl["brand_idx"])
    tbl.rename(columns={"brand_idx": "Brand"}, inplace=True)

for name, tbl in [("mae_table", mae_table),
                  ("rmse_table", rmse_table),
                  ("r2_table", r2_table),
                  ("medae_table", medae_table)]:
    print(f"\n{name}\n" + "-" * len(name))
    print(tbl.to_string(index=False))

Scores in Test on Kaggle: 
- Random Forest: 1192.72192
- Extra Tree: 1182.26039
- Hist Gradient Boosting: 1213.02817
- Hybrid: 1159.51764

# Discussion of Results

The results show that training brand-specific models leads to consistently better performance than using a single global model across all brands. In the per-brand setting, the average MAE across brands is approximately ~1200 (TBD), indicating strong predictive performance given the inherent noise and heterogeneity of used car prices.

Among the individual models, HistGradientBoosting performs worst overall, with an average MAE of (TBD). Random Forest improves upon this baseline, achieving an average MAE of (TBD), while Extra Trees further reduces the error to (TBD) due to better variance reduction and increased randomness. The best performance is achieved by the hybrid ensemble model, which combines Extra Trees and HistGradientBoosting in log-space and reaches the lowest average MAE of (TBD) across brands.

When applying the same algorithms in a single global model trained on all brands, performance degrades slightly for all approaches. On average, the MAE increases by approximately ~20 (TBD) compared to the brand-specific models. This degradation is consistent across all model types and indicates that brand-level specialization provides additional predictive power.

Overall, the hybrid ensemble not only outperforms all individual models in the per-brand framework but also remains the strongest approach in the global setting, making it the best-performing model in the entire project.

# Alignment Between Results and Communicated Objectives

The results align well with the objectives defined at the outset of this analysis. The initial hypothesis was that cars within the same brand exhibit more homogeneous pricing behavior and that training separate models would improve predictive accuracy. The observed MAE improvements for brand-specific models across all approaches (TBD) directly support this assumption.

Furthermore, the superior performance of the hybrid ensemble validates the objective of combining complementary tree-based models. The hybrid model consistently achieves the lowest MAE (TBD), confirming that the additional modeling complexity is justified.

In conclusion, the experimental findings are well aligned with the communicated objectives: brand-specific modeling improves performance, and the hybrid ensemble emerges as the most effective and reliable model for car price prediction in this project.

# Appendix

In [436]:
brand_names = ["Ford", "Mercedes-Benz", "Volkswagen", "Opel", "BMW",
               "Audi", "Toyota", "Škoda", "Hyundai"]

stats = []

for name, y in zip(brand_names, train_y_list):
    # if y is a DataFrame -> take first column
    if isinstance(y, pd.DataFrame):
        y = y.iloc[:, 0]
    # now y should be a Series
    stats.append({
        "Brand": name,
        "count": int(y.shape[0]),
        "avg_price": float(y.mean()),
        "median_price": float(y.median())
    })

stats_df = pd.DataFrame(stats).sort_values("avg_price", ascending=False)

print(stats_df.to_string(index=False))

        Brand  count    avg_price  median_price
Mercedes-Benz   9825 24582.587074       22702.0
         Audi   6165 22879.654015       20251.0
          BMW   6229 22720.178359       20000.0
   Volkswagen   8736 16885.700321       15680.0
        Škoda   3618 14203.493090       12998.0
      Hyundai   2818 12884.833570       12219.5
         Ford  13519 12585.092241       11750.0
       Toyota   3900 12441.260513       10560.0
         Opel   7872 10351.276804        9998.0
